**Reshaping Practices**

| ابزار           | کاربرد                      |
| --------------- | --------------------------- |
| `pivot()`       | Long → Wide                 |
| `pivot_table()` | Long → Wide + Aggregation   |
| `melt()`        | Wide → Long                 |
| `stack()`       | Columns → Index             |
| `unstack()`     | Index → Columns             |
| Reshaping       | مفهوم کلی تغییر ساختار داده |

---

In [19]:
import pandas as pd

##### Exercise — Model Evaluation Dataset
```text
Create the following DataFrame:

import pandas as pd

results = pd.DataFrame({
    "image_id": [
        "img_001", "img_001",
        "img_002", "img_002",
        "img_003", "img_003",
        "img_003"
    ],
    "model": [
        "CNN", "ResNet",
        "CNN", "ResNet",
        "CNN", "CNN",
        "ResNet"
    ],
    "confidence": [
        0.91, 0.95,
        0.82, 0.88,
        0.79, 0.85,
        0.92
    ]
})

Perform the following:

Part 1 — Pivot
Try to convert results from long format to wide format using pivot().
Observe what happens because img_003 + CNN appears more than once.
Explain why pivot() cannot handle this data.

Part 2 — Pivot Table
Use pivot_table() instead.
Use mean as the aggregation function.
Create a second pivot table using max.
Compare the mean and max results.

Part 3 — Melt
Take your mean pivot table and convert it back to long format using melt().
Use:
image_id as id_vars
model as var_name
confidence as value_name

Part 4 — Stack / Unstack
Take the pivoted DataFrame and use stack() to move the model columns into the index.
Use unstack() to transform the result back toward the wide structure.
Inspect the resulting indexes and columns.

Part 5 — Final Understanding

Finally, answer these questions in comments:

# 1. What is the difference between pivot() and pivot_table()?
# 2. When would you use melt()?
# 3. What is the difference between Long and Wide data?
# 4. Why did pivot() fail on this dataset?
# 5. Which operation converts Wide → Long?
# 6. Which operation converts Long → Wide?

In [20]:
results = pd.DataFrame({
    "image_id": [
        "img_001", "img_001",
        "img_002", "img_002",
        "img_003", "img_003",
        "img_003"
    ],
    "model": [
        "CNN", "ResNet",
        "CNN", "ResNet",
        "CNN", "CNN",
        "ResNet"
    ],
    "confidence": [
        0.91, 0.95,
        0.82, 0.88,
        0.79, 0.85,
        0.92
    ]
})

print(results)

  image_id   model  confidence
0  img_001     CNN        0.91
1  img_001  ResNet        0.95
2  img_002     CNN        0.82
3  img_002  ResNet        0.88
4  img_003     CNN        0.79
5  img_003     CNN        0.85
6  img_003  ResNet        0.92


In [21]:
try:
    wide_results = results.pivot(
        index="image_id",
        columns="model",
        values="confidence"
    )

    print(wide_results)

except ValueError as e:
    print("pivot() failed:")
    print(e)

pivot() failed:
Index contains duplicate entries, cannot reshape


In [22]:
# pivot() requires every combination of:
# index + columns
# to be unique.
#
# But img_003 + CNN appears twice:
#
# 0.79
# 0.85
#
# pivot() does not know which value should be placed
# in the img_003 / CNN cell.

In [23]:
wide_results_mean = results.pivot_table(
    index="image_id",
    columns="model",
    values="confidence",
    aggfunc="mean"
)

print(wide_results_mean)

model      CNN  ResNet
image_id              
img_001   0.91    0.95
img_002   0.82    0.88
img_003   0.82    0.92


In [24]:
wide_results_max = results.pivot_table(
    index="image_id",
    columns="model",
    values="confidence",
    aggfunc="max"
)

print(wide_results_max)

model      CNN  ResNet
image_id              
img_001   0.91    0.95
img_002   0.82    0.88
img_003   0.85    0.92


In [25]:
print("Mean:")
print(wide_results_mean)

print("\nMax:")
print(wide_results_max)

# For img_003 + CNN:
#
# mean = (0.79 + 0.85) / 2 = 0.82
# max  = 0.85

Mean:
model      CNN  ResNet
image_id              
img_001   0.91    0.95
img_002   0.82    0.88
img_003   0.82    0.92

Max:
model      CNN  ResNet
image_id              
img_001   0.91    0.95
img_002   0.82    0.88
img_003   0.85    0.92


In [26]:
long_result_mean = (
    wide_results_mean
    .reset_index()
    .melt(
        id_vars="image_id",
        var_name="model",
        value_name="confidence"
    )
)

print(long_result_mean)

  image_id   model  confidence
0  img_001     CNN        0.91
1  img_002     CNN        0.82
2  img_003     CNN        0.82
3  img_001  ResNet        0.95
4  img_002  ResNet        0.88
5  img_003  ResNet        0.92


In [27]:
stacked = wide_results_mean.stack()

print(stacked)

image_id  model 
img_001   CNN       0.91
          ResNet    0.95
img_002   CNN       0.82
          ResNet    0.88
img_003   CNN       0.82
          ResNet    0.92
dtype: float64


In [28]:
print("Index:")
print(stacked.index)

print("\nType:")
print(type(stacked))

Index:
MultiIndex([('img_001',    'CNN'),
            ('img_001', 'ResNet'),
            ('img_002',    'CNN'),
            ('img_002', 'ResNet'),
            ('img_003',    'CNN'),
            ('img_003', 'ResNet')],
           names=['image_id', 'model'])

Type:
<class 'pandas.Series'>


In [29]:
unstacked = stacked.unstack()

print(unstacked)

model      CNN  ResNet
image_id              
img_001   0.91    0.95
img_002   0.82    0.88
img_003   0.82    0.92


In [30]:
print("Index:")
print(unstacked.index)

print("\nColumns:")
print(unstacked.columns)

Index:
Index(['img_001', 'img_002', 'img_003'], dtype='str', name='image_id')

Columns:
Index(['CNN', 'ResNet'], dtype='str', name='model')


In [31]:
# 1. What is the difference between pivot() and pivot_table()?
#
# pivot() requires each index + column combination to be unique.
#
# pivot_table() can handle duplicate combinations by aggregating
# their values using functions such as mean, max, min, sum, etc.


# 2. When would you use melt()?
#
# Use melt() when you want to convert Wide data into Long data.
# It converts multiple value columns into rows.


# 3. What is the difference between Long and Wide data?
#
# Long data stores observations in rows.
#
# Wide data spreads different categories or variables across columns.


# 4. Why did pivot() fail on this dataset?
#
# Because img_003 + CNN appears more than once.
#
# The two confidence values are:
# 0.79
# 0.85
#
# pivot() cannot decide which value belongs in the
# img_003 / CNN cell.


# 5. Which operation converts Wide → Long?
#
# melt()


# 6. Which operation converts Long → Wide?
#
# pivot() or pivot_table()